In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('./orders.csv')

df.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [3]:
df["order_date"] = pd.to_datetime(df["order_date"])
df_sorted = df.sort_values(["customer_id", "order_date"])
df_sorted["inter_order_gap"] = df_sorted.groupby("customer_id")["order_date"].diff()
df.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 646945 entries, 0 to 646944
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        646945 non-null  int64         
 1   order_date      646945 non-null  datetime64[us]
 2   customer_id     646945 non-null  int64         
 3   zip             646945 non-null  int64         
 4   order_status    646945 non-null  str           
 5   payment_method  646945 non-null  str           
 6   device_type     646945 non-null  str           
 7   order_source    646945 non-null  str           
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 39.5 MB


In [5]:
median_gap = df_sorted["inter_order_gap"].dt.days.median()
print(f"Median inter-order gap: {median_gap} days")
print(median_gap-180, median_gap-90)

Median inter-order gap: 144.0 days
-36.0 54.0


so it C) 180 ngày

Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [6]:
df = pd.read_csv("./products.csv")

df.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [7]:
df["gross_margin"] = (
    df["price"] - df["cogs"]
) / df["price"]

df.head()

,product_id,product_name,category,segment,size,color,price,cogs,gross_margin
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875,0.1225
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254,0.4336
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278,0.2871
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954,0.4558
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406,0.1080


In [8]:
avg_margin = df.groupby("segment")["gross_margin"].mean()
highest_segment = avg_margin.idxmax()
highest_value = avg_margin.max()
print(
    f"Phân khúc (segment) có tỷ suất cao nhất là: '{highest_segment}' với giá trị trung bình khoảng {highest_value:.4f}"
)

Phân khúc (segment) có tỷ suất cao nhất là: 'Standard' với giá trị trung bình khoảng 0.3134


so it D) Standard

Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [9]:
df_returns = pd.read_csv("./returns.csv")
df_products = pd.read_csv("./products.csv")
df_merged = pd.merge(df_returns, df_products, on="product_id", how="inner")
streetwear_returns = df_merged[df_merged["category"] == "Streetwear"]
streetwear_returns.head()


,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866
5,RET-000006,59,671,2012-07-19,defective,1,10086.33,SaigonFlex UC-36,Streetwear,Everyday,XL,black,11194.626316,7866.463912
6,RET-000007,67,604,2012-07-16,wrong_size,1,5713.22,SaigonFlex UC-69,Streetwear,Everyday,S,white,6135.298831,4141.940241
7,RET-000008,102,467,2012-07-17,defective,1,9724.09,SaigonFlex UM-72,Streetwear,Balanced,XL,silver,11081.167629,6350.617168


In [10]:
return_reason_counts = streetwear_returns["return_reason"].value_counts()
return_reason_counts

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

so it B) wrong_size

Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [11]:
df_web = pd.read_csv("web_traffic.csv")

df_web.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [12]:
avg_bc_rt = df_web.groupby("traffic_source")["bounce_rate"].mean()

print(f'{avg_bc_rt.idxmin()} have lowest bounce rate is {avg_bc_rt.min()}')

email_campaign have lowest bounce rate is 0.0044584356435643565


so it C) email_campaign

Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [13]:
df_promo = pd.read_csv("./order_items.csv")

df_promo.head()

/var/folders/96/bd16mtp562nd8ycp3zbwzs640000gn/T/ipykernel_95049/1707937637.py:1: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_promo = pd.read_csv("./order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [14]:
df_promo.info()

print(f"{100 - df_promo['promo_id'].isna().mean() * 100:.0f}%")

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  str    
 6   promo_id_2       206 non-null     str    
dtypes: float64(2), int64(3), str(2)
memory usage: 38.2 MB
39%


so it C) 39%

Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [15]:
df_customers = pd.read_csv("./customers.csv")
df_orders = pd.read_csv("./orders.csv")

df_cust_valid = df_customers.dropna(subset=["age_group"])

cust_count_by_age = df_cust_valid.groupby("age_group")["customer_id"].nunique()

df_merged = pd.merge(
    df_orders,
    df_cust_valid[["customer_id", "age_group"]],
    on="customer_id",
    how="inner",
)

order_count_by_age = df_merged.groupby("age_group")["order_id"].nunique()

avg_orders = order_count_by_age / cust_count_by_age

highest_age_group = avg_orders.idxmax()
highest_value = avg_orders.max()

print(
    f"Nhóm tuổi có số đơn trung bình cao nhất là: '{highest_age_group}' với {highest_value:.4f} đơn/KH"
)

Nhóm tuổi có số đơn trung bình cao nhất là: '55+' với 5.4069 đơn/KH


so it A) 55+

Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [16]:
df_orders = pd.read_csv("orders.csv")
df_items = pd.read_csv("order_items.csv")
df_geo = pd.read_csv("geography.csv")

df_items["revenue"] = df_items["quantity"] * df_items["unit_price"]
df_items.head()

/var/folders/96/bd16mtp562nd8ycp3zbwzs640000gn/T/ipykernel_95049/1889994802.py:2: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv("order_items.csv")


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,revenue
0,1,2400,7,1138.22,0.0,NaN,NaN,7967.54
1,2,609,7,10166.25,0.0,NaN,NaN,71163.75
2,3,396,3,11220.33,0.0,NaN,NaN,33660.99
3,4,635,5,10639.25,0.0,NaN,NaN,53196.25
4,6,1935,1,1597.84,0.0,NaN,NaN,1597.84


In [20]:
order_revenue = df_items.groupby("order_id")["revenue"].sum().reset_index()
df_order_zip = pd.merge(
    order_revenue, df_orders[["order_id", "zip"]], on="order_id", how="inner"
)
df_final = pd.merge(df_order_zip, df_geo[["zip", "region"]], on="zip", how="inner")
revenue_by_region = df_final.groupby("region")["revenue"].sum()
best_region = revenue_by_region.idxmax()
best_rev = revenue_by_region.max()
print(f"Vùng có tổng doanh thu cao nhất: {best_region} (Doanh thu: {best_rev:,.2f})")

Vùng có tổng doanh thu cao nhất: East (Doanh thu: 7,637,532,676.20)


so it A) West

Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [18]:
df = pd.read_csv("./orders.csv")
# 1. Filter for orders where the status is 'cancelled'
cancel_ord = df[df["order_status"] == "cancelled"]

# 2. Get the frequency count of each payment method
payment_method_counts = cancel_ord["payment_method"].value_counts()

payment_method_counts


payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

so it A) credit_card

Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [21]:
df_products = pd.read_csv("./products.csv")
df_items = pd.read_csv("./order_items.csv")
df_returns = pd.read_csv("./returns.csv")

valid_sizes = ["S", "M", "L", "XL"]
df_prod_size = df_products[df_products["size"].isin(valid_sizes)][
    ["product_id", "size"]
]

df_ret_joined = pd.merge(df_returns, df_prod_size, on="product_id", how="inner")
return_counts_by_size = df_ret_joined.groupby("size").size()

df_item_joined = pd.merge(df_items, df_prod_size, on="product_id", how="inner")
order_counts_by_size = df_item_joined.groupby("size").size()

return_rates = return_counts_by_size / order_counts_by_size
highest_size = return_rates.idxmax()
highest_rate = return_rates.max()
print(
    f"Kích thước có tỷ lệ trả hàng cao nhất là: '{highest_size}' với tỷ lệ: {highest_rate:.2%}"
)

/var/folders/96/bd16mtp562nd8ycp3zbwzs640000gn/T/ipykernel_95049/548427532.py:2: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv("./order_items.csv")


Kích thước có tỷ lệ trả hàng cao nhất là: 'S' với tỷ lệ: 5.65%


so it A) S

Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?

In [22]:
df_pmt = pd.read_csv("./payments.csv")

df_pmt.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [23]:
total_val_by_inst = df_pmt.groupby("installments")["payment_value"].sum()
orders_by_inst = df_pmt.groupby("installments")["order_id"].nunique()
avg_val_by_inst = total_val_by_inst / orders_by_inst

highest_installments = avg_val_by_inst.idxmax()
highest_avg = avg_val_by_inst.max()
print(
    f"Kế hoạch/số tháng trả góp cao nhất: '{highest_installments}' với mốc thanh toán trung bình: {highest_avg:,.2f}/đơn"
)

Kế hoạch/số tháng trả góp cao nhất: '6' với mốc thanh toán trung bình: 24,446.65/đơn


so it C) 6 kỳ